In [1]:
import math

## SECTION 1 - MANUALLY EDITABLE INPUTS

In [3]:
# ------------------------------------------------------------
# 1.1 Stream 342 component mass flowrates [kg/h]
# ------------------------------------------------------------
m_H2   = 1244.259
m_CO   = 8011.277
m_CO2  = 572.267
m_CH4  = 340.545
m_N2   = 18.857
m_H2S  = 0.009
m_C2H2 = 6.020
m_C2H4 = 73.309
m_C2H6 = 0.314
m_C6H6 = 0.455

# ------------------------------------------------------------
# 1.2 R-401 operating conditions
# ------------------------------------------------------------
T_C = 250.0               # degC
P_bar = 48.963            # bar

# Gas-property inputs at R-401 conditions.
Z = 1.00                  # compressibility factor [-]
gas_viscosity = 2.50e-5   # Pa.s
D_H2S_mix = 1.50e-6       # H2S effective molecular diffusivity in syngas [m2/s]

# ------------------------------------------------------------
# 1.3 Molecular weights [kg/kmol]
# ------------------------------------------------------------
MW_H2   = 2.01588
MW_CO   = 28.0101
MW_CO2  = 44.0095
MW_CH4  = 16.0425
MW_N2   = 28.0134
MW_H2S  = 34.0809
MW_C2H2 = 26.0373
MW_C2H4 = 28.0532
MW_C2H6 = 30.0690
MW_C6H6 = 78.1118
MW_S    = 32.065
MW_ZnO  = 81.38

# ------------------------------------------------------------
# 1.4 Average gas heat capacities [kJ/kg/K]
# Used only for the adiabatic temperature-rise check.
# ------------------------------------------------------------
Cp_H2   = 14.50
Cp_CO   = 1.05
Cp_CO2  = 0.98
Cp_CH4  = 2.69
Cp_N2   = 1.05
Cp_H2S  = 1.05
Cp_C2H2 = 2.02
Cp_C2H4 = 2.05
Cp_C2H6 = 2.37
Cp_C6H6 = 1.61

# ------------------------------------------------------------
# 1.5 Representative commercial ZnO sorbent
# BASF Zn 9201 used as representative physical basis.
# ------------------------------------------------------------
pellet_diameter = 0.004       # m; nominal 4 mm extrudate
ZnO_mass_fraction = 0.95      # kg ZnO/kg sorbent
bulk_density = 1150.0         # kg sorbent/m3 packed bed
bed_voidage = 0.40            # engineering assumption [-]

# Nominal sulfur capacity and breakthrough utilisation
nominal_sulfur_capacity = 0.20      # kg S/kg sorbent
breakthrough_utilisation = 0.70     # fraction of nominal capacity used at breakthrough

# ------------------------------------------------------------
# 1.6 Service and protection criteria
# ------------------------------------------------------------
service_period_years = 2.0
breakthrough_H2S_ppmv = 0.10
allowable_dP_bar = 0.10

# ------------------------------------------------------------
# 1.7 Vessel diameter selection settings
# ------------------------------------------------------------
# The continuous minimum diameter satisfying the Ergun limit is calculated.
# The selected diameter is then rounded UP to the next practical increment.
diameter_rounding_increment = 0.05  # m
diameter_search_min = 0.20          # m
diameter_search_max = 3.00          # m

# ------------------------------------------------------------
# 1.8 Sulfidation heat
# ZnO + H2S -> ZnS + H2O(g)
# ------------------------------------------------------------
deltaH_sulfidation = -76.7   # kJ/mol H2S, approximate

# ------------------------------------------------------------
# 1.9 Constants
# ------------------------------------------------------------
R_kPa = 8.314462618          # kPa.m3/(kmol.K)
R_SI = 8.314462618           # Pa.m3/(mol.K)

T_normal_K = 273.15
P_normal_kPa = 101.325

## SECTION 2 - ORGANISE STREAM DATA

In [5]:
mass_flow = {
    "H2": m_H2,
    "CO": m_CO,
    "CO2": m_CO2,
    "CH4": m_CH4,
    "N2": m_N2,
    "H2S": m_H2S,
    "C2H2": m_C2H2,
    "C2H4": m_C2H4,
    "C2H6": m_C2H6,
    "C6H6": m_C6H6,
}

MW = {
    "H2": MW_H2,
    "CO": MW_CO,
    "CO2": MW_CO2,
    "CH4": MW_CH4,
    "N2": MW_N2,
    "H2S": MW_H2S,
    "C2H2": MW_C2H2,
    "C2H4": MW_C2H4,
    "C2H6": MW_C2H6,
    "C6H6": MW_C6H6,
}

Cp = {
    "H2": Cp_H2,
    "CO": Cp_CO,
    "CO2": Cp_CO2,
    "CH4": Cp_CH4,
    "N2": Cp_N2,
    "H2S": Cp_H2S,
    "C2H2": Cp_C2H2,
    "C2H4": Cp_C2H4,
    "C2H6": Cp_C2H6,
    "C6H6": Cp_C6H6,
}

T_K = T_C + 273.15
P_kPa = P_bar * 100.0
P_Pa = P_bar * 1.0e5

service_period_h = (
    service_period_years
    * 365.0
    * 24.0
)

## SECTION 3 - STREAM MOLAR FLOW AND H2S CONCENTRATION

In [7]:
molar_flow = {
    component: mass_flow[component] / MW[component]
    for component in mass_flow
}  # kmol/h

total_mass_flow = sum(mass_flow.values())
total_molar_flow = sum(molar_flow.values())

mixture_MW = (
    total_mass_flow / total_molar_flow
)

mole_fraction = {
    component: molar_flow[component] / total_molar_flow
    for component in molar_flow
}

y_H2S = mole_fraction["H2S"]

H2S_inlet_ppmv = (
    y_H2S * 1.0e6
)

required_H2S_removal_fraction = (
    1.0
    - breakthrough_H2S_ppmv / H2S_inlet_ppmv
)

if required_H2S_removal_fraction <= 0.0:
    raise ValueError(
        "The inlet H2S concentration is already at or below the "
        "specified breakthrough criterion."
    )

## SECTION 4 - SULFUR LOADING AND ZnO INVENTORY

In [9]:
# Conservative design assumption:
# all sulfur entering R-401 is assumed to be captured over the service period.

S_mass_fraction_in_H2S = (
    MW_S / MW_H2S
)

sulfur_mass_flow = (
    m_H2S
    * S_mass_fraction_in_H2S
)  # kg S/h

sulfur_load_service_period = (
    sulfur_mass_flow
    * service_period_h
)  # kg S

working_sulfur_capacity = (
    nominal_sulfur_capacity
    * breakthrough_utilisation
)  # kg S/kg sorbent

required_sorbent_mass = (
    sulfur_load_service_period
    / working_sulfur_capacity
)  # kg sorbent

required_bed_volume = (
    required_sorbent_mass
    / bulk_density
)  # m3

# Stoichiometric reference only
pure_ZnO_stoich_S_capacity = (
    MW_S / MW_ZnO
)

commercial_stoich_S_capacity = (
    ZnO_mass_fraction
    * pure_ZnO_stoich_S_capacity
)

## SECTION 5 - GAS FLOW AND DENSITY AT R-401 CONDITIONS

In [11]:
normal_gas_flow = (
    total_molar_flow
    * R_kPa
    * T_normal_K
    / P_normal_kPa
)  # Nm3/h

actual_gas_flow = (
    total_molar_flow
    * R_kPa
    * T_K
    * Z
    / P_kPa
)  # m3/h

gas_density = (
    total_mass_flow
    / actual_gas_flow
)  # kg/m3

## SECTION 6 - HYDRAULIC MODEL

In [13]:
def bed_geometry(diameter):
    """Return area, packed height and superficial gas velocity."""
    area = (
        math.pi * diameter**2 / 4.0
    )

    height = (
        required_bed_volume / area
    )

    superficial_velocity = (
        actual_gas_flow
        / 3600.0
        / area
    )

    return (
        area,
        height,
        superficial_velocity,
    )


def ergun_pressure_drop(diameter):
    """
    Calculate total packed-bed pressure drop [bar]
    for a specified vessel internal diameter [m].
    """
    area, height, u_s = bed_geometry(diameter)

    eps = bed_voidage
    dp = pellet_diameter
    mu = gas_viscosity
    rho = gas_density

    viscous_term = (
        150.0
        * mu
        * (1.0 - eps)**2
        / (eps**3 * dp**2)
        * u_s
    )

    inertial_term = (
        1.75
        * rho
        * (1.0 - eps)
        / (eps**3 * dp)
        * u_s**2
    )

    dP_dL = (
        viscous_term
        + inertial_term
    )  # Pa/m

    total_dP_bar = (
        dP_dL
        * height
        / 1.0e5
    )

    return (
        total_dP_bar,
        viscous_term,
        inertial_term,
    )

## SECTION 7 - DETERMINE MINIMUM AND SELECTED DIAMETER

In [15]:
if ergun_pressure_drop(diameter_search_max)[0] > allowable_dP_bar:
    raise ValueError(
        "No diameter within the specified search range satisfies "
        "the allowable pressure-drop criterion."
    )

D_low = diameter_search_min
D_high = diameter_search_max

for _ in range(200):
    D_mid = 0.5 * (D_low + D_high)

    if ergun_pressure_drop(D_mid)[0] > allowable_dP_bar:
        D_low = D_mid
    else:
        D_high = D_mid

minimum_ID_continuous = D_high

selected_ID = (
    math.ceil(
        minimum_ID_continuous
        / diameter_rounding_increment
        - 1.0e-12
    )
    * diameter_rounding_increment
)

while (
    ergun_pressure_drop(selected_ID)[0]
    > allowable_dP_bar
):
    selected_ID += diameter_rounding_increment

## SECTION 8 - FINAL BED GEOMETRY AND PRESSURE DROP

In [17]:
(
    cross_sectional_area,
    bed_height,
    superficial_velocity,
) = bed_geometry(selected_ID)

(
    bed_pressure_drop_bar,
    dP_dL_viscous,
    dP_dL_inertial,
) = ergun_pressure_drop(selected_ID)

L_over_D = (
    bed_height / selected_ID
)

pressure_drop_pass = (
    bed_pressure_drop_bar
    <= allowable_dP_bar
)

actual_GHSV = (
    normal_gas_flow
    / required_bed_volume
)  # h^-1

empty_bed_contact_time = (
    required_bed_volume
    / actual_gas_flow
    * 3600.0
)  # s

## SECTION 9 - EXTERNAL H2S MASS-TRANSFER CHECK

In [19]:
particle_Re = (
    gas_density
    * superficial_velocity
    * pellet_diameter
    / gas_viscosity
)

Schmidt_number = (
    gas_viscosity
    / (
        gas_density
        * D_H2S_mix
    )
)

Sherwood_number = (
    2.0
    + 1.1
    * particle_Re**0.6
    * Schmidt_number**(1.0 / 3.0)
)

gas_film_coefficient = (
    Sherwood_number
    * D_H2S_mix
    / pellet_diameter
)  # m/s

external_area_per_bed_volume = (
    6.0
    * (1.0 - bed_voidage)
    / pellet_diameter
)  # m2/m3 bed

total_external_particle_area = (
    external_area_per_bed_volume
    * required_bed_volume
)  # m2

total_gas_concentration = (
    P_Pa
    / (
        Z
        * R_SI
        * T_K
    )
)  # mol/m3

H2S_bulk_concentration = (
    y_H2S
    * total_gas_concentration
)  # mol/m3

H2S_inlet_mol_s = (
    molar_flow["H2S"]
    * 1000.0
    / 3600.0
)  # mol/s

required_H2S_removal_mol_s = (
    H2S_inlet_mol_s
    * required_H2S_removal_fraction
)

required_average_H2S_flux = (
    required_H2S_removal_mol_s
    / total_external_particle_area
)  # mol/m2/s

required_film_concentration_drop = (
    required_average_H2S_flux
    / gas_film_coefficient
)  # mol/m3

film_drop_fraction = (
    required_film_concentration_drop
    / H2S_bulk_concentration
)

estimated_surface_H2S_concentration = (
    H2S_bulk_concentration
    - required_film_concentration_drop
)

external_transfer_capacity_ratio = (
    1.0 / film_drop_fraction
)

## SECTION 10 - ISOTHERMAL ASSUMPTION CHECK

In [21]:
H2S_mol_per_h = (
    molar_flow["H2S"]
    * 1000.0
)

maximum_heat_release_kJ_h = (
    H2S_mol_per_h
    * abs(deltaH_sulfidation)
)

maximum_heat_release_W = (
    maximum_heat_release_kJ_h
    * 1000.0
    / 3600.0
)

gas_heat_capacity_flow = sum(
    mass_flow[component]
    * Cp[component]
    for component in mass_flow
)  # kJ/h/K

maximum_adiabatic_deltaT = (
    maximum_heat_release_kJ_h
    / gas_heat_capacity_flow
)  # K

## SECTION 11 - SERVICE-LIFE CHECK

In [23]:
usable_sulfur_capacity = (
    required_sorbent_mass
    * working_sulfur_capacity
)

calculated_service_life_h = (
    usable_sulfur_capacity
    / sulfur_mass_flow
)

calculated_service_life_years = (
    calculated_service_life_h
    / (365.0 * 24.0)
)

full_nominal_capacity_life_years = (
    required_sorbent_mass
    * nominal_sulfur_capacity
    / sulfur_mass_flow
    / (365.0 * 24.0)
)

## SECTION 12 - PRINT RESULTS

In [25]:
print("\n" + "=" * 78)
print("R-401 ZnO GUARD BED - PROCESS DESIGN")
print("=" * 78)

print("\nDESIGN BASIS")
print(f"R-401 inlet temperature              = {T_C:,.1f} degC")
print(f"R-401 inlet pressure                 = {P_bar:,.3f} bar")
print(f"H2S inlet concentration              = {H2S_inlet_ppmv:,.4f} ppmv")
print(f"H2S breakthrough criterion          = {breakthrough_H2S_ppmv:,.3f} ppmv")
print(f"Required service period              = {service_period_years:,.2f} years")

print("\nSORPTION DESIGN")
print(f"Sulfur load over service period      = {sulfur_load_service_period:,.2f} kg S")
print(f"Nominal sulfur capacity              = {100*nominal_sulfur_capacity:,.1f} wt% S")
print(f"Breakthrough utilisation             = {100*breakthrough_utilisation:,.1f} %")
print(f"Working sulfur capacity              = {100*working_sulfur_capacity:,.1f} wt% S")
print(f"Required ZnO sorbent inventory       = {required_sorbent_mass:,.1f} kg")
print(f"Required packed-bed volume           = {required_bed_volume:,.3f} m3")

print("\nREACTOR GEOMETRY")
print(f"Continuous minimum ID from Ergun     = {minimum_ID_continuous:,.3f} m")
print(f"Selected practical vessel ID         = {selected_ID:,.3f} m")
print(f"Packed-bed height                    = {bed_height:,.3f} m")
print(f"Bed L/D ratio                        = {L_over_D:,.3f}")
print(f"ZnO pellet diameter                  = {1000*pellet_diameter:,.1f} mm")

print("\nHYDRAULICS")
print(f"Actual gas volumetric flow           = {actual_gas_flow:,.2f} m3/h")
print(f"Gas density                          = {gas_density:,.3f} kg/m3")
print(f"Superficial gas velocity             = {superficial_velocity:,.4f} m/s")
print(f"Packed-bed pressure drop             = {bed_pressure_drop_bar:,.4f} bar")
print(f"Allowable pressure drop              = {allowable_dP_bar:,.4f} bar")
print(f"Pressure-drop check                  = {'PASS' if pressure_drop_pass else 'FAIL'}")
print(f"Resulting GHSV                       = {actual_GHSV:,.0f} h^-1")
print(f"Empty-bed contact time               = {empty_bed_contact_time:,.2f} s")

print("\nEXTERNAL H2S MASS TRANSFER")
print(f"Particle Reynolds number             = {particle_Re:,.2f}")
print(f"Schmidt number                       = {Schmidt_number:,.3f}")
print(f"Sherwood number                      = {Sherwood_number:,.2f}")
print(f"Gas-film coefficient                 = {gas_film_coefficient:,.5f} m/s")
print(f"Required film concentration drop     = {100*film_drop_fraction:,.3f} % of bulk H2S")
print(f"External transfer capacity ratio     = {external_transfer_capacity_ratio:,.1f} x required")
print(f"Estimated C_surface / C_bulk         = {estimated_surface_H2S_concentration/H2S_bulk_concentration:,.4f}")

print("\nTHERMAL CHECK")
print(f"Maximum sulfidation heat release     = {maximum_heat_release_W:,.4f} W")
print(f"Maximum adiabatic temperature rise   = {maximum_adiabatic_deltaT:,.6f} K")

print("\nSERVICE-LIFE CHECK")
print(f"Calculated design service life       = {calculated_service_life_years:,.2f} years")
print(f"Life if full nominal capacity used   = {full_nominal_capacity_life_years:,.2f} years")

print("\nFINAL R-401 PROCESS DESIGN OUTPUTS")
print("-" * 78)
print(f"ZnO sorbent inventory                = {required_sorbent_mass:,.1f} kg")
print(f"Packed-bed volume                    = {required_bed_volume:,.3f} m3")
print(f"Vessel internal diameter             = {selected_ID:,.3f} m")
print(f"Packed-bed height                    = {bed_height:,.3f} m")
print(f"Bed L/D                              = {L_over_D:,.3f}")
print(f"ZnO pellet diameter                  = {1000*pellet_diameter:,.1f} mm")
print(f"Superficial gas velocity             = {superficial_velocity:,.4f} m/s")
print(f"Packed-bed pressure drop             = {bed_pressure_drop_bar:,.4f} bar")
print("=" * 78)


R-401 ZnO GUARD BED - PROCESS DESIGN

DESIGN BASIS
R-401 inlet temperature              = 250.0 degC
R-401 inlet pressure                 = 48.963 bar
H2S inlet concentration              = 0.2806 ppmv
H2S breakthrough criterion          = 0.100 ppmv
Required service period              = 2.00 years

SORPTION DESIGN
Sulfur load over service period      = 148.35 kg S
Nominal sulfur capacity              = 20.0 wt% S
Breakthrough utilisation             = 70.0 %
Working sulfur capacity              = 14.0 wt% S
Required ZnO sorbent inventory       = 1,059.7 kg
Required packed-bed volume           = 0.921 m3

REACTOR GEOMETRY
Continuous minimum ID from Ergun     = 0.906 m
Selected practical vessel ID         = 0.950 m
Packed-bed height                    = 1.300 m
Bed L/D ratio                        = 1.368
ZnO pellet diameter                  = 4.0 mm

HYDRAULICS
Actual gas volumetric flow           = 835.96 m3/h
Gas density                          = 12.282 kg/m3
Superficial gas veloc